# Audio Segmentation Pipeline

This Colab provides a streamlined pipeline for extracting labeled audio segments from Google Cloud Storage (GCS) using JSONL annotations and exporting them back to GCS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/common/chirp_and_gemini_segment_audio.ipynb)

### Core Functions
1. **Manifest Retrieval**: Fetches target file lists from GCS. Only annotated files are processed.
2. **Local Audio Caching**: Downloads source audio into a local cache. On subsequent runs, it skips downloads if the file is already present locally.
3. **Ground Truth Slicing**: Extract audio segments as defined by the start/end timestamps in the manifest. Optionally adds silence to start and end of audio.
4. **Automated Export**:
    * Uploads processed FLAC segments to GCS organized by `example_id`.
    * Generates and uploads a `batch_manifest.jsonl` containing metadata for all segments (GCS paths, offsets, and durations) to facilitate downstream ASR evaluation.

In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    ffmpeg-python

In [ ]:
# @title Imports and environment configuration
import json
import sys
from pathlib import Path
from urllib.parse import urlparse

import ffmpeg
from loguru import logger

# @markdown ### User Configuration
# @markdown Please enter your GCP project and bucket details below:
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = "one_hour_pilot"  # @param {type:"string"}
# @markdown Enable modifications (e.g. silence padding, future normalization):
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown If False, skip segments that already exist in GCS:
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

assert GCP_PROJECT_ID, "Please enter your GCP project ID"
assert GCS_BUCKET, "Please enter your GCS bucket name"
assert PROJECT_NAME, "PROJECT_NAME must be provided."

SOURCE_MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/manifests/{PROJECT_NAME}_transcriptions.json"
)
GCS_OUTPUT_PREFIX = f"segmented_audio/{PROJECT_NAME}_audio"

LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segments"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
from google.colab import auth
from google.cloud import storage

# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Helper functions
import base64
import io
import json
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import HTML, display
import numpy as np
import scipy.signal
import soundfile as sf


def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Ensures the audio file from GCS is available locally in CACHE_DIR."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")

    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    # Ensure the parent directory exists before attempting to download
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        bucket = gcs_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(str(local_path))
    return str(local_path)


def cleanup_gcs_output(output_prefix: str) -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    if blobs:
        logger.info(
            f"Cleaning existing files from gs://{GCS_BUCKET}/{output_prefix}"
        )
        bucket.delete_blobs(blobs)


def run_pilot_pipeline(
    segmentation_name: str = "default",
    apply_merging: bool = False,
    max_duration_s: float = 60.0,
) -> None:
    """Orchestrates extraction, preserving native audio specs. Fails fast on errors."""
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Determine Output Prefix
    if apply_merging:
        current_output_prefix = f"{GCS_OUTPUT_PREFIX}/labels_merged"
    else:
        current_output_prefix = f"{GCS_OUTPUT_PREFIX}/{segmentation_name}"

    if not AUDIO_PREPROCESSING:
        current_output_prefix += "_raw"

    if OVERWRITE_EXISTING:
        cleanup_gcs_output(current_output_prefix)

    # 2. Load the JSON or JSONL manifest from GCS
    parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
    m_bucket = gcs_client.bucket(parsed_manifest.netloc)
    m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
    content = m_blob.download_as_text()

    manifest_data = []
    # Robust parsing handling both single JSON array and JSONL
    try:
        manifest_data = json.loads(content)
        logger.info(f"Loaded JSON manifest with {len(manifest_data)} entries.")
    except json.JSONDecodeError as e:
        if "Extra data" in str(e):
            manifest_data = [
                json.loads(line)
                for line in content.strip().split("\n")
                if line.strip()
            ]
            logger.info(
                f"Loaded JSONL manifest with {len(manifest_data)} entries."
            )
        else:
            raise e

    # Optional: Apply segment merging before processing
    if apply_merging:
        logger.info("Applying segment merging simulation...")
        manifest_data = simulate_segment_merging(
            manifest_data, max_duration_s=max_duration_s
        )

    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    output_bucket = gcs_client.bucket(GCS_BUCKET)
    final_manifest_entries = []

    # 3. Process segments (Optimized with in-memory NumPy slicing)
    for gcs_audio_path, segments in files_to_process.items():
        local_src_path = ensure_local_gcs_audio(gcs_audio_path)

        # Read the full audio file into memory once
        try:
            audio_data, native_sr = sf.read(local_src_path, always_2d=True)
            native_channels = audio_data.shape[1]
            logger.info(
                f"Loaded {Path(gcs_audio_path).name} ({native_sr}Hz, {native_channels}ch)"
            )
        except Exception as e:
            logger.error(f"Failed to read {local_src_path} with soundfile: {e}")
            continue

        example_id = Path(gcs_audio_path).stem

        for i, seg in enumerate(segments):
            seg_id = f"{i:03d}"
            transcription = seg.get("text", "")
            if not transcription:
                logger.info(
                    f"Skipping because no text. Of category: {seg.get('category', '')}"
                )
                continue

            start_s = seg["offset"]
            duration_s = seg["duration"]
            filename = f"{example_id}__seg{seg_id}.flac"
            blob_name = f"{current_output_prefix}/{example_id}/{filename}"
            output_blob = output_bucket.blob(blob_name)

            total_duration = (
                duration_s + 3 if AUDIO_PREPROCESSING else duration_s
            )

            # Skip if file exists and we aren't overwriting
            if not OVERWRITE_EXISTING and output_blob.exists():
                logger.info(
                    f"  > Skipping segment {seg_id} (already exists in GCS)"
                )
                output
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "example_id": example_id,
                        "offset": start_s,
                        "duration": total_duration,
                        "segment_id": seg_id,
                        "text": transcription,
                    }
                )
                continue

            # Calculate sample indices for slicing
            start_sample = int(start_s * native_sr)
            end_sample = int((start_s + duration_s) * native_sr)

            # Ensure we don't exceed audio length
            end_sample = min(end_sample, audio_data.shape[0])

            # Slice the audio in memory
            sliced_audio = audio_data[start_sample:end_sample]

            if AUDIO_PREPROCESSING:
                # Pad with silence: 1s pre, 2s post
                silence_pre = np.zeros(
                    (native_sr, native_channels), dtype=audio_data.dtype
                )
                silence_post = np.zeros(
                    (2 * native_sr, native_channels), dtype=audio_data.dtype
                )
                processed_audio = np.concatenate(
                    [silence_pre, sliced_audio, silence_post], axis=0
                )
            else:
                processed_audio = sliced_audio

            local_slice_path = Path(SEGMENTS_DIR) / filename

            # Write the slice as FLAC
            try:
                sf.write(
                    local_slice_path, processed_audio, native_sr, format="FLAC"
                )
            except Exception as e:
                logger.error(f"Failed to write slice {filename}: {e}")
                continue

            # Upload to GCS
            logger.info(f"  > Uploading segment {seg_id} to GCS...")
            output_blob.upload_from_filename(str(local_slice_path))
            # Clean up local slice file after upload
            local_slice_path.unlink()

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "example_id": example_id,
                    "offset": start_s,
                    "duration": total_duration,
                    "segment_id": seg_id,
                    "text": transcription,
                }
            )

        # Clean up local source audio file after all its segments are processed
        if local_src_path and Path(local_src_path).exists():
            Path(local_src_path).unlink()

    # 4. Final Manifest
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{current_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))
    logger.info(
        f"Done. {len(final_manifest_entries)} entries in manifest for {current_output_prefix}."
    )


def merge_segments(segments, max_duration_s: float = 60.0):
    """Merges consecutive segments that have text transcripts, respecting break labels and max duration."""
    segments = sorted(segments, key=lambda x: x["offset"])
    merged = []
    current_merge = None

    # Define categories that break the chain
    break_categories = ["unintelligible", "pii/id", "dtmf"]

    for seg in segments:
        has_text = bool(seg.get("text", "").strip())
        category = seg.get("category", "").strip().lower()

        # Check if category is a break label
        is_break_label = any(cat in category for cat in break_categories)

        if has_text and not is_break_label:
            if current_merge is None:
                current_merge = {
                    "audio_filepath": seg["audio_filepath"],
                    "example_id": seg.get(
                        "example_id", Path(seg["audio_filepath"]).stem
                    ),
                    "offset": seg["offset"],
                    "duration": seg["duration"],
                    "segment_id": [
                        seg.get("segment_id", "")
                    ],  # Store as list of IDs
                    "text": seg["text"],
                }
            else:
                # Calculate potential new duration
                potential_end_time = max(
                    current_merge["offset"] + current_merge["duration"],
                    seg["offset"] + seg["duration"],
                )
                potential_duration = (
                    potential_end_time - current_merge["offset"]
                )

                # Check if adding this segment exceeds max duration
                if potential_duration <= max_duration_s:
                    # Extend current merged segment
                    current_merge["duration"] = potential_duration
                    current_merge["text"] += " " + seg["text"]
                    current_merge["segment_id"].append(
                        seg.get("segment_id", "")
                    )
                else:
                    # Exceeds max duration, close current merge and start new one
                    merged.append(current_merge)
                    current_merge = {
                        "audio_filepath": seg["audio_filepath"],
                        "example_id": seg.get(
                            "example_id", Path(seg["audio_filepath"]).stem
                        ),
                        "offset": seg["offset"],
                        "duration": seg["duration"],
                        "segment_id": [seg.get("segment_id", "")],
                        "text": seg["text"],
                    }
        else:
            # Break label or no text breaks the consecutive merge chain
            if current_merge is not None:
                merged.append(current_merge)
                current_merge = None

    if current_merge is not None:
        merged.append(current_merge)

    return merged


def simulate_segment_merging(manifest_data, max_duration_s: float = 60.0):
    """Simulates merging neighboring transcribed segments without unintelligible sections."""
    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    all_merged_segments = []
    for path, segments in files_to_process.items():
        merged = merge_segments(segments, max_duration_s=max_duration_s)
        all_merged_segments.extend(merged)

    logger.info(
        f"Simulated merging: {len(manifest_data)} original segments -> {len(all_merged_segments)} merged segments."
    )
    return all_merged_segments


def visualize_audio_with_segments(
    audio_array: np.ndarray,
    segments: list,
    target_sr: int = 16000,
    container_id: str = "waveform",
    wave_color: str = "tomato",
    progress_color: str = "firebrick",
    title_text: str = "Audio Configuration",
) -> None:
    """Creates interactable waveform charts representing slice regions natively via WaveSurfer.js."""
    if audio_array.ndim > 1 and audio_array.shape[1] > 1:
        audio_array = np.mean(audio_array, axis=1)
    elif audio_array.ndim == 2 and audio_array.shape[1] == 1:
        audio_array = audio_array.flatten()

    wav_io = io.BytesIO()
    sf.write(wav_io, audio_array, target_sr, format="WAV", subtype="PCM_16")
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    # FIX: Create DOM elements for content to force WaveSurfer to render HTML instead of escaping it
    regions_js = "".join(
        [
            f"""
        var div_{i} = document.createElement('div');
        div_{i}.innerHTML = `{seg["label"]}`;
        wsRegions.addRegion({{start: {seg["start"]}, end: {seg["end"]}, content: div_{i}, color: '{seg["color"]}'}});
        """
            for i, seg in enumerate(segments)
        ]
    )

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>{title_text} ({len(segments)} segments)</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: '{wave_color}',
            progressColor: '{progress_color}',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))

In [ ]:
# @title Create the pilot segments and manifest file
run_pilot_pipeline()